<a href="https://colab.research.google.com/github/Simiformes/zadania-rekrutacyjne/blob/main/Zadanie2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y torchao
!pip install -U ms-swift datasets rouge_score sacrebleu "qwen_vl_utils>=0.0.14"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 105.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
from datasets import load_dataset

raw_dataset = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information", split="train")

filtered_data = []

for item in raw_dataset:
    inst = item.get('instruction', '') or ''
    inp = item.get('input', '') or ''
    out = item.get('output', '') or ''

    full_input = f"{inst}\n{inp}".strip() if inp else inst.strip()

    if len(full_input) == 0 or len(out) == 0:
        continue
    if len(out) < 20:
        continue
    if len(full_input) + len(out) > 1500:
        continue

    filtered_data.append({"query": full_input, "response": out})

print(f"Liczba przykładów przed filtracją: {len(raw_dataset)}")
print(f"Liczba przykładów po filtracji: {len(filtered_data)}")

test_data = filtered_data[:100]
train_data = filtered_data[100:]

with open("train_data.jsonl", "w", encoding="utf-8") as f:
    for entry in train_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

README.md:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

medical_meadow_wikidoc_patient_info.json: reconstructing file:   0%|          |  0.00B / 3.49MB            

medical_meadow_wikidoc_patient_info.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

Liczba przykładów przed filtracją: 5942
Liczba przykładów po filtracji: 5701


In [16]:
import os
os.environ['USE_HF'] = '1'

import numpy as np
import sacrebleu
import torch
from rouge_score import rouge_scorer
from swift.infer_engine import TransformersEngine, RequestConfig, InferRequest

model = 'qwen/Qwen3.5-2B'
engine_base = TransformersEngine(model, torch_dtype=torch.float16)
request_config = RequestConfig(max_tokens=512, temperature=0.2, repetition_penalty=1.15)

def evaluate_model(engine, test_samples, max_eval=100):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    predictions = []
    references = []
    word_counts = []

    samples_to_eval = test_samples[:max_eval]

    for item in samples_to_eval:
        prompt = item['query']
        ref = item['response']

        infer_requests = [InferRequest(messages=[{'role': 'user', 'content': prompt}])]

        resp_list = engine.infer(infer_requests, request_config)
        pred = resp_list[0].choices[0].message.content.strip()

        predictions.append(pred)
        references.append(ref)
        word_counts.append(len(pred.split()))

    r1, r2, rl = [], [], []
    for p, r in zip(predictions, references):
        scores = scorer.score(r, p)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)

    bleu = sacrebleu.corpus_bleu(predictions, [[r for r in references]]).score

    results = {
        'ROUGE-1': round(float(np.mean(r1)), 6),
        'ROUGE-2': round(float(np.mean(r2)), 6),
        'ROUGE-L': round(float(np.mean(rl)), 6),
        'BLEU': round(float(bleu), 6),
        'Średnia liczba słów': round(float(np.mean(word_counts)), 2)
    }

    return results, predictions, references

results_before, preds_before, refs = evaluate_model(engine_base, test_data, max_eval=50)
print("Wyniki przed tuningiem:", results_before)

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000


Wyniki przed tuningiem: {'ROUGE-1': 0.219078, 'ROUGE-2': 0.041505, 'ROUGE-L': 0.103766, 'BLEU': 1.109728, 'Średnia liczba słów': 281.14}


In [6]:
!swift sft \
    --model qwen/Qwen3.5-2B \
    --use_hf true \
    --tuner_type lora \
    --dataset train_data.jsonl \
    --torch_dtype float16 \
    --attn_impl sdpa \
    --num_train_epochs 1 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --target_modules all-linear \
    --max_length 768 \
    --split_dataset_ratio 0 \
    --eval_strategy no \
    --logging_steps 10 \
    --save_steps 500 \
    --save_total_limit 1 \
    --dataloader_num_workers 4 \
    --dataset_num_proc 4 \
    --output_dir output

run sh: `/usr/bin/python3 /usr/local/lib/python3.13/dist-packages/swift/cli/sft.py --model qwen/Qwen3.5-2B --use_hf true --tuner_type lora --dataset train_data.jsonl --torch_dtype float16 --attn_impl sdpa --num_train_epochs 1 --per_device_train_batch_size 2 --gradient_accumulation_steps 4 --learning_rate 1e-4 --lora_rank 8 --lora_alpha 32 --target_modules all-linear --max_length 768 --split_dataset_ratio 0 --eval_strategy no --logging_steps 10 --save_steps 500 --save_total_limit 1 --dataloader_num_workers 4 --dataset_num_proc 4 --output_dir output`
[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: qwen/Qwen3.5-2B
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files: 100% 12/12 [00:00<00:00, 1521.33it/s]
Download complete:           |  0.

In [17]:
import glob

checkpoint_dirs = glob.glob("output/v2-20261029-201034/checkpoint-701")
latest_checkpoint = sorted(checkpoint_dirs)[-1]
print(f"Wczytywanie LoRA z: {latest_checkpoint}")

engine_lora = TransformersEngine(model, adapters=[latest_checkpoint])

results_after, preds_after, _ = evaluate_model(engine_lora, test_data, max_eval=50)
print("Wyniki po tuningu:", results_after)

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: qwen/Qwen3.5-2B


Wczytywanie LoRA z: output/v2-20261029-201034/checkpoint-701


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[INFO:swift] Setting torch_dtype: torch.bfloat16
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/models--qwen--Qwen3.

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] Successfully loaded output/v2-20261029-201034/checkpoint-701/args.json.
[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000


Wyniki po tuningu: {'ROUGE-1': 0.320298, 'ROUGE-2': 0.092446, 'ROUGE-L': 0.198062, 'BLEU': 5.813261, 'Średnia liczba słów': 69.12}


In [5]:
import pandas as pd
results_before = {'ROUGE-1': 0.219078, 'ROUGE-2': 0.041505, 'ROUGE-L': 0.103766, 'BLEU': 1.109728, 'Średnia liczba słów': 281.14}
results_after = {'ROUGE-1': 0.320298, 'ROUGE-2': 0.092446, 'ROUGE-L': 0.198062, 'BLEU': 5.813261, 'Średnia liczba słów': 69.12}
df_results = pd.DataFrame([results_before, results_after], index=["Przed tuningiem", "Po tuningu"])
print("=== ZESTAWIENIE WYNIKÓW EWALUACJI ===")
display(df_results)

#print("\n=== PRZYKŁADOWE ODPOWIEDZI MODELU ===")
#for i in range(2):
 #   print(f"\n--- PRZYKŁAD {i+1} ---")
  #  print(f"PYTANIE: {test_data[1+i]['query']}")
   # print(f"ODPOWIEDŹ WZORCOWA (REFERENCE): {refs[i]}")
    #print(f"ODPOWIEDŹ PRZED TUNINGIEM: {preds_before[i]}")
    #print(f"ODPOWIEDŹ PO TUNINGU: {preds_after[i+1]}")

=== ZESTAWIENIE WYNIKÓW EWALUACJI ===


,ROUGE-1,ROUGE-2,ROUGE-L,BLEU,Średnia liczba słów
Przed tuningiem,0.219078,0.041505,0.103766,1.109728,281.14
Po tuningu,0.320298,0.092446,0.198062,5.813261,69.12
